## 환경 준비

아래 셀은 이 노트북에 필요한 Python 패키지가 설치되어 있는지 확인하고,
없으면 자동으로 설치한다. 이미 설치되어 있으면 빠르게 스킵된다.
터미널에서 미리 `uv sync`를 했다면 이 셀은 아무것도 설치하지 않는다.


In [ ]:
# === 의존성 자동 설치 (이미 설치되어 있으면 빠르게 스킵됨) ===
import subprocess, sys

_IMPORT_NAME_OVERRIDES = {
    "scikit-learn": "sklearn",
    "python-dateutil": "dateutil",
    "beautifulsoup4": "bs4",
}


def _ensure_packages(*packages):
    """누락된 패키지만 설치. 이미 있으면 스킵."""
    missing = []
    for pkg in packages:
        name = pkg.split(">=")[0].split("==")[0].split("[")[0]
        import_name = _IMPORT_NAME_OVERRIDES.get(name, name.replace("-", "_"))
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All packages already installed ✓")

_ensure_packages(
    "pandas", "numpy", "matplotlib", "scikit-learn",
    "pydantic", "python-dateutil", "rich", "tqdm",
)


# 에이전트 시스템 평가(Evaluation)

agent 시스템은 데모 질문 하나에 잘 답했다고 끝나지 않는다. 어떤 질문 유형에서 강한지, 어떤 질문에서 retrieval은 맞지만 synthesis가 약한지, abstain은 얼마나 정밀하게 하는지 수치로 봐야 개선 방향을 잡을 수 있다. 이 노트북은 baseline과 agent workflow를 같은 데이터셋에 반복 실행하고, 대표 지표를 읽는 법까지 함께 익히는 튜토리얼이다.

## 학습 목표
- 평가 데이터셋이 어떤 스키마로 구성되는지 이해한다.
- `answer_correctness`, `retrieval_hit_rate`, `grounding_pass_rate`, `abstain_precision`, `latency`, `average_steps`를 읽을 수 있다.
- baseline과 agent workflow의 차이를 단일 숫자가 아니라 지표 조합으로 해석할 수 있다.
- radar chart가 "속도-품질-안전성"의 트레이드오프를 어떻게 시각화하는지 설명할 수 있다.


## 개념 설명

평가(evaluation)는 모델을 심판하기 위한 절차가 아니라, 시스템의 약한 고리를 찾기 위한 계측(instrumentation)이다. 특히 agent 시스템은 retrieval, planning, tool use, synthesis, verification, fallback이 함께 움직이므로 "정답률 하나"만 보면 어디를 고쳐야 할지 알 수 없다. 그래서 이 notebook은 단계별로 읽을 수 있는 지표를 쓴다.

**목적**
- 왜 여러 metric이 필요한지 먼저 정리한다.

**핵심 로직**
- 정답 겹침은 answer quality를 본다.
- retrieval hit rate는 근거 검색이 맞았는지 본다.
- grounding pass는 verifier 기준의 신뢰성을 본다.
- abstain precision은 abstain이 얼마나 정확했는지 본다.

**결과 해석 가이드**
- 좋은 시스템은 모든 숫자가 동시에 최고가 되기보다, 자신의 목적에 맞는 균형을 가진다.

**💡 면접 포인트**
- "Evaluation은 leaderboard보다 diagnosis에 가깝다"고 말하면 설계 의도를 잘 드러낼 수 있다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## 왜 evaluation이 중요한가

먼저 평가 데이터셋을 읽는다. 데이터셋은 실험의 기준면(base plane)이다. 질문 타입, 기대 상태(answered/abstained), gold answer, expected source가 명확해야 retrieval과 answer quality를 분리해서 볼 수 있다.

**목적**
- 평가에 사용할 샘플 구조를 확인한다.

**핵심 로직**
- `load_eval_dataset()`는 기본적으로 `data/eval/eval_dataset.json`을 읽는다.
- 노트북에서는 이를 DataFrame으로 바꿔 question type과 기대 상태를 함께 확인한다.

**주요 파라미터/변수**
- `id`: 평가 샘플 식별자
- `question_type`: 기대되는 질의 유형
- `expected_status`: answered 또는 abstained
- `expected_sources`: retrieval hit rate의 기준 source

**실제 소스 코드: load_eval_dataset() — src/evaluator.py**
```python
def load_eval_dataset(dataset_path: Path | None = None) -> list[dict[str, Any]]:
    paths = get_paths()
    source = dataset_path or paths.eval_dir / "eval_dataset.json"
    return list(read_json(source))
```

**코드 읽기 포인트**
- 평가셋 로더를 작게 유지해, 데이터 교체가 코드 로직을 흔들지 않도록 했다.
- 실무에서도 evaluation pipeline은 로더가 복잡한 것보다 스키마가 명확한 것이 더 중요하다.

**결과 해석 가이드**
- `Total questions`가 많다고 좋은 평가가 되는 것은 아니다. 중요한 것은 query type과 failure mode가 균형 있게 섞여 있는가이다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.evaluator import load_eval_dataset, run_evaluation_suite

dataset = load_eval_dataset()
dataset_frame = pd.DataFrame(dataset)
print(f'Total questions: {len(dataset_frame)}')
dataset_frame[['id', 'question_type', 'expected_status', 'question']].head(12)

## 평가 데이터셋 설계(Designing evaluation datasets)

아래 분포 차트는 데이터셋이 특정 질문 유형에 치우치지 않았는지 빠르게 보여준다. agent 시스템 평가는 특히 중요하다. 예를 들어 `insufficient_evidence_risk`가 거의 없으면 abstain 로직이 좋아 보이는지 나빠 보이는지 판단할 근거가 사라진다.

**목적**
- 질문 유형 분포가 실험 해석에 어떤 영향을 주는지 본다.

**핵심 로직**
- 질문 수 분포를 보면 metric 평균이 어떤 유형에 끌려가는지 추정할 수 있다.

**결과 해석 가이드**
- 막대가 균형에 가까울수록 평균 metric을 해석하기 쉬워진다.
- 특정 유형이 적으면 평균보다 type별 breakdown을 더 신뢰해야 한다.


## 구현

분포 차트는 단순한 EDA처럼 보이지만, 사실 평가 리스크를 미리 드러내는 장치다. 어떤 type이 적은지, 이후 notebook에서 어떤 failure case를 더 보강해야 하는지 여기서 감이 잡힌다.

**결과 해석 가이드**
- `count`가 작은 type은 요약 평균에서 영향력이 약하므로, 나중에 별도 breakdown을 꼭 봐야 한다.


In [ ]:
distribution = dataset_frame['question_type'].value_counts().sort_index()
ax = distribution.plot(kind='bar', color='#4C78A8', title='Evaluation Dataset Distribution by Query Type')
ax.set_xlabel('query_type')
ax.set_ylabel('question count')
plt.tight_layout()
plt.show()
distribution.reset_index().rename(columns={'index': 'question_type', 'question_type': 'count'})

## 지표(metrics)

이제 실제 평가를 돌린다. 이 셀을 읽기 전에 각 metric이 무엇을 재는지 정확히 이해해 두는 것이 좋다.

**목적**
- baseline과 agent workflow를 같은 기준으로 반복 실행한다.

**핵심 로직**
- `score_answer()`는 gold answer와 predicted answer의 token F1을 계산한다.
- `retrieval_hit_rate()`는 gold source가 top-k에 들어왔는지 0 또는 1로 본다.
- `run_evaluation_suite()`는 baseline과 agent workflow를 각각 실행해 하나의 결과표로 합친다.

**주요 파라미터/변수**
- `repeats=2`: 같은 실험을 여러 번 돌려 latency와 step 수를 평균 낼 수 있게 한다.
- `persist_outputs=True`: 결과를 JSON 리포트로 저장한다.

**실제 소스 코드: score_answer() — src/evaluator.py**
```python
def score_answer(predicted_answer: str, gold_answer: str, predicted_status: str, expected_status: str) -> float:
    if expected_status == "abstained":
        return 1.0 if predicted_status == "abstained" else 0.0
    if predicted_status == "abstained":
        return 0.0
    return token_f1(predicted_answer, gold_answer)
```

**실제 소스 코드: retrieval_hit_rate() — src/evaluator.py**
```python
def retrieval_hit_rate(retrieved_docs: list[dict[str, Any]], expected_sources: list[str]) -> float:
    if not expected_sources:
        return 1.0 if retrieved_docs else 0.0
    sources = {doc["source"] for doc in retrieved_docs}
    return 1.0 if any(source in sources for source in expected_sources) else 0.0
```

**실제 소스 코드: token_f1() — src/utils.py**
```python
def token_f1(prediction: str, gold: str) -> float:
    prediction_tokens = content_tokens(prediction)
    gold_tokens = content_tokens(gold)
    if not prediction_tokens or not gold_tokens:
        return 0.0
    overlap = len(set(prediction_tokens) & set(gold_tokens))
    if overlap == 0:
        return 0.0
    precision = overlap / len(set(prediction_tokens))
    recall = overlap / len(set(gold_tokens))
    return round(2 * precision * recall / (precision + recall), 3)
```

**코드 읽기 포인트**
- `answer_correctness`: token F1으로 gold와 prediction의 핵심 토큰 겹침을 본다.
- `retrieval_hit_rate`: expected source가 검색 결과에 들어왔는지만 보므로 0 또는 1이다.
- `grounding_pass_rate`: verifier가 grounded라고 판정한 비율이다.
- `abstain_precision`: abstain한 것 중 실제로 abstain해야 했던 비율이다.
- `latency`: 속도, `average_steps`: reasoning/processing depth의 거친 proxy다.

**결과 해석 가이드**
- `answer_correctness`가 높아도 `grounding_pass_rate`가 낮으면 그럴듯하지만 위험한 시스템일 수 있다.
- `retrieval_hit_rate`가 높은데 correctness가 낮으면 synthesis나 planning 문제일 가능성이 크다.

**💡 면접 포인트**
- "Metric은 한 방향으로만 올리면 안 된다. correctness, grounding, abstention은 서로 긴장 관계에 있다"고 말할 수 있다.


In [ ]:
results, summary = run_evaluation_suite(repeats=2, persist_outputs=True)
summary

## baseline과 agent 비교

이제 raw result와 question-type breakdown을 같이 본다. 이 단계의 목적은 평균 summary를 보기 전에, 어떤 유형에서 차이가 발생했는지를 먼저 읽는 것이다. 평균은 편리하지만, multi-hop과 summary를 하나의 숫자로 섞어 버리면 원인 분석이 어려워진다.

**목적**
- 시스템별 평균뿐 아니라 question type별 특성도 함께 본다.

**핵심 로직**
- `evaluate_system()`는 각 질문마다 결과 record를 만들고, failure type까지 붙인다.
- `summarize_results()`는 system별 평균 metric만 간단히 모은다.

**주요 파라미터/변수**
- `predicted_question_type`: classifier 품질을 간접적으로 볼 수 있는 필드
- `average_steps`: workflow가 얼마나 많은 단계를 거쳤는지 보여주는 값

**실제 소스 코드: evaluate_system() — src/evaluator.py**
```python
def evaluate_system(
    system: SystemName,
    dataset: list[dict[str, Any]] | None = None,
    trace_dir: Path | None = None,
    repeats: int = 3,
) -> pd.DataFrame:
    evaluation_set = dataset or load_eval_dataset()
    retriever = build_demo_index(persist=False)
    rows: list[dict[str, Any]] = []

    for run_id in range(1, repeats + 1):
        for sample in evaluation_set:
            start = time.perf_counter()
            if system == "baseline":
                result = run_baseline_rag(sample["question"], retriever)
                predicted_question_type = None
                grounding_pass = False
            else:
                trace_path = None
                if trace_dir is not None:
                    trace_path = trace_dir / f"{sample['id']}_run{run_id}.json"
                result = run_workflow(sample["question"], retriever, trace_path=trace_path)
                predicted_question_type = result["query_type"]
                grounding_pass = result["verification_result"].is_grounded

            latency = time.perf_counter() - start
            record = EvaluationRecord(
                system=system,
                question_id=sample["id"],
                run_id=run_id,
                question=sample["question"],
                expected_question_type=sample["question_type"],
                predicted_question_type=predicted_question_type,
                expected_status=sample.get("expected_status", "answered"),
                predicted_status=result["final_status"],
                final_answer=result["final_answer"],
                answer_correctness=score_answer(
                    result["final_answer"],
                    sample["gold_answer"],
                    result["final_status"],
                    sample.get("expected_status", "answered"),
                ),
                retrieval_hit_rate=retrieval_hit_rate(result["retrieved_docs"], sample.get("expected_sources", [])),
                grounding_pass=grounding_pass,
                abstained=result["final_status"] == "abstained",
                abstain_precision=0.0,
                latency_seconds=round(latency, 4),
                average_steps=float(len(result.get("trace", []))),
                failure_type="",
            ).to_dict()
            record["expected_sources"] = sample.get("expected_sources", [])
            record["trace"] = result.get("trace", [])
            record["errors"] = result.get("errors", [])
            record["citations"] = [
                citation.get("source", "")
                for citation in result.get("citations", [])
                if isinstance(citation, dict) and citation.get("source")
            ]
            record["failure_type"] = classify_failure(record)
            record["abstain_precision"] = 1.0 if (
                record["abstained"] and record["expected_status"] == "abstained"
            ) else 0.0
            rows.append(record)

    frame = pd.DataFrame(rows)
    if not frame.empty:
        frame["grounding_pass_rate"] = frame["grounding_pass"].astype(float)
    else:
        frame["grounding_pass_rate"] = []
    return frame
```

**실제 소스 코드: summarize_results() — src/evaluator.py**
```python
def summarize_results(results: pd.DataFrame) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()

    rows: list[dict[str, Any]] = []
    for system, frame in results.groupby("system"):
        abstained = frame[frame["abstained"]]
        abstain_precision = (
            float((abstained["expected_status"] == "abstained").mean()) if not abstained.empty else 0.0
        )
        rows.append(
            {
                "system": system,
                "answer_correctness": frame["answer_correctness"].mean(),
                "retrieval_hit_rate": frame["retrieval_hit_rate"].mean(),
                "grounding_pass_rate": frame["grounding_pass_rate"].mean(),
                "abstain_precision": abstain_precision,
                "latency": frame["latency_seconds"].mean(),
                "average_steps": frame["average_steps"].mean(),
            }
        )
    return pd.DataFrame(rows).round(3)
```

**코드 읽기 포인트**
- `evaluate_system()`은 baseline과 agent를 같은 dataset에 반복 실행해 비교 가능성을 맞춘다.
- agent workflow 쪽은 trace, errors, citations까지 record에 포함해 나중 failure analysis에 재사용한다.
- `summarize_results()`는 system별 평균만 남겨 dashboard 역할을 한다.

**결과 해석 가이드**
- question type breakdown에서 특정 type만 급격히 낮으면 평균보다 그 type의 trace를 우선 보는 것이 좋다.
- `average_steps`가 높은 것은 계산 비용이 늘었다는 뜻이지만, 복잡한 질문에서 더 안전한 판단을 했다는 뜻일 수도 있다.


In [ ]:
question_type_breakdown = (
    results.groupby(['system', 'expected_question_type'])[[
        'answer_correctness',
        'retrieval_hit_rate',
        'grounding_pass_rate',
        'abstain_precision',
        'latency_seconds',
        'average_steps',
    ]]
    .mean()
    .round(3)
)

display(summary)
display(question_type_breakdown)

## 실험

radar chart는 여러 metric을 한눈에 비교할 때 유용하지만, 읽는 법을 모르면 오해하기 쉽다. 이 notebook은 speed를 `1 / latency`로 뒤집어 다른 품질 metric과 같은 방향으로 맞춘 뒤, baseline과 agent workflow를 한 평면에 겹쳐 그린다.

**목적**
- 다중 metric을 한 눈에 비교하는 시각화 읽는 법을 익힌다.

**핵심 로직**
- answer correctness, retrieval hit, grounding pass, abstain precision, speed score를 함께 본다.

**결과 해석 가이드**
- 곡선이 바깥으로 클수록 그 metric이 상대적으로 좋다.
- 다만 한 축이 커졌다고 전체가 좋아졌다고 볼 수는 없다. 예를 들어 speed를 높이려다 grounding이 떨어질 수 있다.


In [ ]:
radar = summary.set_index('system')[['answer_correctness', 'retrieval_hit_rate', 'grounding_pass_rate', 'abstain_precision']].copy()
radar['speed_score'] = 1.0 / summary.set_index('system')['latency'].clip(lower=0.001)
radar['speed_score'] = radar['speed_score'] / radar['speed_score'].max()
radar_metrics = list(radar.columns)
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})
for system, row in radar.iterrows():
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, label=system)
    ax.fill(angles, values, alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics)
ax.set_title('Baseline vs Agent Workflow Radar View')
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.show()
radar.round(3)

## 결과 해석

마지막 delta 표는 agent_workflow에서 baseline을 뺀 값이다. 여기서 중요한 것은 "무엇이 증가했고 무엇이 감소했는가"를 trade-off로 읽는 것이다. 예를 들어 latency가 증가했더라도 grounding pass와 abstain precision이 크게 좋아졌다면, 안전성 확보를 위해 감수할 수 있는 비용일 수 있다.

**목적**
- metric 변화량을 우선순위와 연결한다.

**결과 해석 가이드**
- 운영 안전성이 중요한 시스템이면 `grounding_pass_rate`와 `abstain_precision` 개선을 더 높게 볼 수 있다.
- 검색 품질은 그대로인데 correctness만 낮다면 synthesis를, correctness는 높은데 grounding이 낮다면 verifier/fallback을 우선 개선한다.

**💡 면접 포인트**
- "어떤 metric을 올릴 것인가"는 제품 목적의 문제다. 빠른 내부 검색 도우미와 보수적인 연구 assistant는 최적점이 다르다.


In [ ]:
metric_deltas = summary.set_index('system').loc['agent_workflow'] - summary.set_index('system').loc['baseline']
metric_deltas.to_frame(name='agent_minus_baseline').round(3)

## 핵심 정리

이 노트북을 통해 evaluation은 단순 정답률 체크가 아니라 시스템 진단 도구라는 점을 확인했다. `answer_correctness`는 답변 품질, `retrieval_hit_rate`는 근거 검색, `grounding_pass_rate`는 verifier 기준의 신뢰도, `abstain_precision`은 안전한 거절의 정밀도를 본다. 따라서 하나의 metric만 보고 시스템을 평가하면 어디를 개선해야 할지 알기 어렵다.

baseline과 agent workflow 비교에서는 보통 속도(latency)와 안전성(grounding, abstention) 사이의 긴장이 드러난다. agent workflow는 더 많은 단계를 거치므로 느려질 수 있지만, 그 대가로 더 안전한 답변과 보수적인 abstain을 제공한다.

**💡 면접 포인트**
- "평가는 leaderboard가 아니라 diagnosis다. retrieval, synthesis, verification을 분리해서 봐야 한다."
- "Radar chart는 여러 metric의 균형을 보여주지만, 제품 목적에 맞는 trade-off 해석이 함께 따라야 한다."
- "좋은 agent 시스템은 correctness만 높은 것이 아니라, 근거와 abstention 정책도 함께 관리한다."
